## End-to-End Machine Learning Assignment

## Diamond Price Prediction using KNN and Streamlit Deployment

#### Part 1: Machine Learning Pipeline

#### Step 1 – Load the dataset using Pandas

In [1]:
import pandas as pd
import numpy as np

In [3]:
DATA_PATH = "diamonds.csv"
df = pd.read_csv(r"D:\MACHINE LEARNING\archive\diamonds.csv")

In [4]:
df.head()

,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [5]:
df.shape

(53940, 11)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  53940 non-null  int64  
 1   carat       53940 non-null  float64
 2   cut         53940 non-null  object 
 3   color       53940 non-null  object 
 4   clarity     53940 non-null  object 
 5   depth       53940 non-null  float64
 6   table       53940 non-null  float64
 7   price       53940 non-null  int64  
 8   x           53940 non-null  float64
 9   y           53940 non-null  float64
 10  z           53940 non-null  float64
dtypes: float64(6), int64(2), object(3)
memory usage: 4.5+ MB


In [9]:
df.columns

Index(['Unnamed: 0', 'carat', 'cut', 'color', 'clarity', 'depth', 'table',
       'price', 'x', 'y', 'z'],
      dtype='object')

In [15]:
df.dtypes.to_frame("Data type")

,Data type
Unnamed: 0,int64
carat,float64
cut,object
color,object
clarity,object
depth,float64
table,float64
price,int64
x,float64
y,float64


In [16]:
df.isnull().sum().to_frame("Missing values")

,Missing values
Unnamed: 0,0
carat,0
cut,0
color,0
clarity,0
depth,0
table,0
price,0
x,0
y,0


#### Step 2 – Define input features (X) and target variable (price)

In [23]:
if "price" not in df.columns:
    raise ValueError("The dataset must contain a 'price' column.")

X = df.drop(columns=["price"])
y = df["price"]

In [24]:
X.shape

(53940, 9)

In [25]:
y.shape

(53940,)

In [26]:
X.columns.tolist()

['carat', 'cut', 'color', 'clarity', 'depth', 'table', 'x', 'y', 'z']

#### Step 3 – Split into training and testing sets (75:25)

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print(f"Training rows: {len(X_train)} ({len(X_train)/len(df):.0%})")
print(f"Testing rows:  {len(X_test)} ({len(X_test)/len(df):.0%})")

Training rows: 40455 (75%)
Testing rows:  13485 (25%)


#### Step 4 – Data preprocessing

- Encode categorical features with `OneHotEncoder`.
- Scale numerical features with `StandardScaler`.
- Combine preprocessing and modeling in a Scikit-learn `Pipeline`. The transformer is fitted on training data only, preventing data leakage.

In [38]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_features = X.select_dtypes(include=["number"]).columns.tolist()

In [39]:
categorical_features

['cut', 'color', 'clarity']

In [40]:
numerical_features

['carat', 'depth', 'table', 'x', 'y', 'z']

In [41]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numerical", StandardScaler(), numerical_features)
    ],
    remainder="drop"
)

#### Step 5 – Train a KNN Regressor

Train a K-Nearest Neighbors regressor with `n_neighbors=5` inside the preprocessing pipeline.

In [42]:
from sklearn.neighbors import KNeighborsRegressor

knn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", KNeighborsRegressor(n_neighbors=5))
])

knn_pipeline.fit(X_train, y_train)
print("KNN regression pipeline trained successfully.")

KNN regression pipeline trained successfully.


#### Step 6 – Evaluate the model

Evaluate the held-out test data using:
- **MAE:** Mean Absolute Error
- **RMSE:** Root Mean Squared Error
- **R²:** Coefficient of determination

Lower MAE/RMSE indicate smaller errors; higher R² indicates more target variance explained.

In [48]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = knn_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

evaluation = pd.DataFrame({
    "Metric": ["Mean Absolute Error (MAE)", "Root Mean Squared Error (RMSE)", "R² Score"],
    "Value": [mae, rmse, r2]
})

In [49]:
evaluation

,Metric,Value
0,Mean Absolute Error (MAE),408.161884
1,Root Mean Squared Error (RMSE),785.867518
2,R² Score,0.960687


In [51]:
print(f"MAE:  {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²:   {r2:.4f}")

MAE:  408.16
RMSE: 785.87
R²:   0.9607


#### Step 7 – Save the trained pipeline using Pickle

Save the complete pipeline (preprocessing plus model) so future predictions use the same transformations.

In [52]:
import pickle

MODEL_PATH = "knn_diamond_price_pipeline.pkl"
with open(MODEL_PATH, "wb") as model_file:
    pickle.dump(knn_pipeline, model_file)

print("Saved trained pipeline:", MODEL_PATH)

Saved trained pipeline: knn_diamond_price_pipeline.pkl


##### Verify the saved pipeline

In [53]:
with open(MODEL_PATH, "rb") as model_file:
    loaded_pipeline = pickle.load(model_file)

sample_X = X_test.head()
sample_predictions = loaded_pipeline.predict(sample_X)

verification = sample_X.copy()
verification["Actual price"] = y_test.loc[sample_X.index].values
verification["Predicted price"] = sample_predictions

In [54]:
verification

,carat,cut,color,clarity,depth,table,x,y,z,Actual price,Predicted price
1388,0.24,Ideal,G,VVS1,62.1,56.0,3.97,4.00,2.47,559,586.4
50052,0.58,Very Good,F,VVS2,60.0,57.0,5.44,5.42,3.26,2201,2427.6
41645,0.40,Ideal,E,VVS2,62.1,55.0,4.76,4.74,2.95,1238,1083.4
42377,0.43,Premium,E,VVS2,60.8,57.0,4.92,4.89,2.98,1304,1196.2
17244,1.55,Ideal,E,SI2,62.3,55.0,7.44,7.37,4.61,6901,9642.2
